# ⚡ Taller de Clase — Performance Testing
## Sesión 8: Pruebas de Rendimiento

**Universidad:** Universitaria de Colombia  
**Docente:** Mg. Sergio Alejandro Burbano Mena  
**Materia:** Pruebas de Software | 7mo Semestre

---

| Parte | Tipo | Tiempo |
|-------|------|--------|
| 1 | Load Testing | 25 min |
| 2 | Stress Testing | 20 min |
| 3 | Análisis y Reporte | 15 min |

**Contexto:** La API del Sistema de Notas de la Universitaria acaba de recibir una nueva versión del módulo de búsqueda. Antes de subir a producción, el equipo QA debe ejecutar una suite completa de performance testing.

---

> 💡 **Instrucciones:** Lee cada celda antes de ejecutarla. Completa los `TODO` con tu implementación.

---
## ⚙️ Setup — Instalar dependencias

In [ ]:
# ══════════════════════════════════════════════════════════════
#  CELDA 0 — Instalación de librerías
#
#  requests   : hacer peticiones HTTP
#  locust     : framework de performance testing
#  matplotlib : graficar los resultados
#  pandas     : analizar los CSV de resultados
# ══════════════════════════════════════════════════════════════

!pip install requests locust matplotlib pandas --quiet

import time           # para medir tiempos
import threading      # para simular usuarios concurrentes
import statistics     # para calcular percentiles
import random         # para variar los datos de prueba
import csv            # para leer los reportes
import json           # para trabajar con respuestas API

import requests       # para hacer peticiones HTTP
import pandas as pd   # para analizar datos
import matplotlib.pyplot as plt  # para graficar
import matplotlib.patches as mpatches

print('✅ Setup completo.')
print('   Locust, requests, matplotlib, pandas — listos.')

---
## 🏗️ Sección 1 — Sistema simulado (API de Notas)

Como no tenemos un servidor real, usamos un simulador que replica el comportamiento de una API REST.  
El simulador tiene tiempos de respuesta **realistas** y puede ser configurado para simular diferentes condiciones.

In [ ]:
# ══════════════════════════════════════════════════════════════
#  CELDA 1 — API simulada del Sistema de Notas
#
#  Esta clase simula una API REST que:
#  - Tiene tiempos de respuesta variables y realistas
#  - Se degrada bajo alta carga (simula cuellos de botella)
#  - Tiene un endpoint lento que será nuestro problema a detectar
# ══════════════════════════════════════════════════════════════

class SimuladorAPISistemaNotas:
    """
    Simula la API REST del Sistema de Notas universitario.
    
    Cada método simula un endpoint HTTP con tiempos de respuesta
    realistas. Bajo alta carga, los tiempos aumentan.
    """
    
    def __init__(self):
        # Contador de peticiones activas simultáneas
        # (simula la carga del servidor)
        self._peticiones_activas = 0
        self._lock = threading.Lock()  # para thread safety
        self._total_peticiones = 0
        self._errores = 0
    
    def _simular_respuesta(self, tiempo_base_ms: float, factor_carga: bool = True) -> dict:
        """
        Simula el tiempo de respuesta de un endpoint.
        
        Args:
            tiempo_base_ms: tiempo base en milisegundos bajo carga normal
            factor_carga:   si True, el tiempo aumenta con la carga
        
        Returns:
            dict con el resultado de la petición
        """
        with self._lock:
            self._peticiones_activas += 1
            carga = self._peticiones_activas
            self._total_peticiones += 1
        
        try:
            # El tiempo de respuesta aumenta con la carga activa
            # Esto simula cómo un servidor real se degrada bajo presión
            if factor_carga and carga > 10:
                # Factor de degradación: aumenta exponencialmente con la carga
                factor = 1 + (carga - 10) * 0.15
                tiempo_base_ms = tiempo_base_ms * factor
            
            # Agregamos variabilidad aleatoria (simula el jitter de la red)
            variacion = random.uniform(0.7, 1.3)
            tiempo_real = tiempo_base_ms * variacion
            
            # Simulamos que el 0.5% de peticiones fallan aleatoriamente
            if random.random() < 0.005:
                with self._lock:
                    self._errores += 1
                time.sleep(tiempo_real / 1000)  # convertir ms a segundos
                return {'status': 500, 'error': 'Internal Server Error', 'tiempo_ms': tiempo_real}
            
            # Simular el tiempo de proceso
            time.sleep(tiempo_real / 1000)
            
            return {'status': 200, 'tiempo_ms': tiempo_real}
        
        finally:
            # Siempre decrementamos, incluso si hubo error
            with self._lock:
                self._peticiones_activas -= 1
    
    # ── ENDPOINTS SIMULADOS ─────────────────────────────────
    
    def get_health(self) -> dict:
        """GET /api/health — El endpoint más rápido"""
        return self._simular_respuesta(tiempo_base_ms=15, factor_carga=False)
    
    def listar_estudiantes(self) -> dict:
        """GET /api/estudiantes — Rápido con caché"""
        return self._simular_respuesta(tiempo_base_ms=85)
    
    def consultar_notas_estudiante(self, estudiante_id: str) -> dict:
        """GET /api/notas/{id} — Query a BD con índice"""
        return self._simular_respuesta(tiempo_base_ms=120)
    
    def buscar_notas(self, query: str) -> dict:
        """
        GET /api/notas/buscar?q={query}
        ENDPOINT LENTO — query sin índice en BD
        Este es el cuello de botella que debemos detectar
        """
        # Base de 800ms porque hace un full table scan en la BD
        return self._simular_respuesta(tiempo_base_ms=800)
    
    def registrar_nota(self, estudiante_id: str, materia: str, nota: float) -> dict:
        """POST /api/notas — Escritura en BD con transacción"""
        return self._simular_respuesta(tiempo_base_ms=200)
    
    def generar_reporte(self, semestre: str) -> dict:
        """
        GET /api/reportes/{semestre}
        ENDPOINT MUY LENTO — genera reporte completo
        """
        return self._simular_respuesta(tiempo_base_ms=2500)
    
    def get_stats(self) -> dict:
        """Estadísticas del simulador"""
        return {
            'total_peticiones': self._total_peticiones,
            'errores': self._errores,
            'error_rate_pct': round(self._errores / max(1, self._total_peticiones) * 100, 2)
        }
    
    def reset_stats(self):
        """Reinicia los contadores de estadísticas"""
        with self._lock:
            self._total_peticiones = 0
            self._errores = 0


# Creamos la instancia global del simulador
api = SimuladorAPISistemaNotas()

print('✅ Simulador de API creado')
print()
print('📡 Endpoints disponibles:')
print('   GET  /api/health              → ~15ms')
print('   GET  /api/estudiantes         → ~85ms')
print('   GET  /api/notas/{id}          → ~120ms')
print('   GET  /api/notas/buscar?q=X    → ~800ms (⚠️ lento!)')
print('   POST /api/notas               → ~200ms')
print('   GET  /api/reportes/{semestre} → ~2500ms (⚠️ muy lento!)')

---
## 📊 Herramientas de Medición

Antes de hacer los tests, necesitamos funciones para **medir y analizar** los resultados.

In [ ]:
# ══════════════════════════════════════════════════════════════
#  CELDA 2 — Funciones de medición y análisis
# ══════════════════════════════════════════════════════════════

def medir_endpoint(funcion_endpoint, n_veces: int = 50) -> dict:
    """
    Ejecuta un endpoint N veces de forma SECUENCIAL y mide los tiempos.
    Útil para establecer el baseline de rendimiento.
    
    Args:
        funcion_endpoint: función que llama al endpoint
        n_veces:          número de peticiones a realizar
    
    Returns:
        Diccionario con estadísticas de los tiempos medidos
    """
    tiempos = []   # lista donde guardaremos todos los tiempos
    errores = 0    # contador de peticiones fallidas
    
    for i in range(n_veces):
        # Medimos el tiempo real de ejecución (wall clock time)
        inicio = time.perf_counter()  # más preciso que time.time()
        respuesta = funcion_endpoint()
        fin = time.perf_counter()
        
        # Calculamos el tiempo en milisegundos
        tiempo_ms = (fin - inicio) * 1000
        tiempos.append(tiempo_ms)
        
        if respuesta['status'] != 200:
            errores += 1
    
    # Calculamos estadísticas
    tiempos.sort()  # ordenar para calcular percentiles
    
    return {
        'n_peticiones': n_veces,
        'errores': errores,
        'error_rate': round(errores / n_veces * 100, 2),
        'min_ms': round(min(tiempos), 1),
        'max_ms': round(max(tiempos), 1),
        'avg_ms': round(statistics.mean(tiempos), 1),
        'p50_ms': round(tiempos[int(n_veces * 0.50)], 1),
        'p90_ms': round(tiempos[int(n_veces * 0.90)], 1),
        'p95_ms': round(tiempos[int(n_veces * 0.95)], 1),
        'p99_ms': round(tiempos[min(int(n_veces * 0.99), n_veces-1)], 1),
        'tiempos_raw': tiempos  # para graficar después
    }


def imprimir_resultados(nombre_endpoint: str, resultados: dict, sla_p95_ms: float = 500.0):
    """
    Imprime los resultados de medición de forma legible.
    Indica claramente si se cumple el SLA.
    """
    p95 = resultados['p95_ms']
    cumple_sla = p95 <= sla_p95_ms
    estado = '✅ CUMPLE SLA' if cumple_sla else '❌ VIOLA SLA'
    
    print(f'\n{'═'*55}')
    print(f'  Endpoint: {nombre_endpoint}')
    print(f'{'═'*55}')
    print(f'  Peticiones:  {resultados["n_peticiones"]:>6}  |  Errores: {resultados["errores"]} ({resultados["error_rate"]}%)')
    print(f'  Min:         {resultados["min_ms"]:>6.1f} ms')
    print(f'  p50 (median):{resultados["p50_ms"]:>6.1f} ms')
    print(f'  p90:         {resultados["p90_ms"]:>6.1f} ms')
    print(f'  p95:         {resultados["p95_ms"]:>6.1f} ms  → {estado} (SLA: {sla_p95_ms}ms)')
    print(f'  p99:         {resultados["p99_ms"]:>6.1f} ms')
    print(f'  Max:         {resultados["max_ms"]:>6.1f} ms')
    print(f'{'═'*55}')


print('✅ Funciones de medición listas.')
print('   medir_endpoint(fn, n) → estadísticas completas')
print('   imprimir_resultados(nombre, resultados, sla) → reporte')

---
# 🏋️ PARTE 1 — Load Testing (25 minutos)

## Objetivo
Verificar que la API del Sistema de Notas cumple el SLA bajo **carga esperada** (50 usuarios concurrentes).

## SLA definido por el cliente
- `p95 < 500ms` para todos los endpoints críticos
- `Error rate < 1%`
- `Throughput > 50 req/s`

## Tu tarea
1. Medir el baseline (sin carga)
2. Implementar el test de carga con usuarios concurrentes
3. Analizar los resultados

In [ ]:
# ══════════════════════════════════════════════════════════════
#  CELDA 3 — PASO 1: Baseline — medir SIN carga
#  (secuencial, un usuario a la vez)
# ══════════════════════════════════════════════════════════════

print('📊 Midiendo BASELINE (sin carga — secuencial)...')
print('   Esto establece el rendimiento base del sistema.')
print()

api.reset_stats()

# Medimos cada endpoint de forma secuencial (baseline)
endpoints_baseline = {
    'GET /api/health':          lambda: api.get_health(),
    'GET /api/estudiantes':     lambda: api.listar_estudiantes(),
    'GET /api/notas/{id}':      lambda: api.consultar_notas_estudiante('2024-001'),
    'GET /api/notas/buscar':    lambda: api.buscar_notas('pruebas de software'),
    'POST /api/notas':          lambda: api.registrar_nota('2024-001', 'INF-701', 4.5),
}

resultados_baseline = {}

for nombre, funcion in endpoints_baseline.items():
    print(f'   Midiendo {nombre}...')
    resultados_baseline[nombre] = medir_endpoint(funcion, n_veces=30)
    imprimir_resultados(nombre, resultados_baseline[nombre])

print('\n✅ Baseline completo.')
print('   Guarda estos números — los compararás con el test de carga.')

In [ ]:
# ══════════════════════════════════════════════════════════════
#  CELDA 4 — PASO 2: Load Test con usuarios concurrentes
#
#  Implementa la función load_test() que:
#  1. Crea N hilos (usuarios virtuales)
#  2. Cada hilo ejecuta peticiones durante `duracion_s` segundos
#  3. Recolecta todos los tiempos de respuesta
#  4. Calcula las estadísticas finales
# ══════════════════════════════════════════════════════════════

def load_test(
    funcion_endpoint,
    n_usuarios: int,
    duracion_s: float,
    ramp_up_s: float = 5.0
) -> dict:
    """
    Ejecuta un load test con N usuarios concurrentes.
    
    Args:
        funcion_endpoint: función que llama al endpoint a probar
        n_usuarios:       número de usuarios concurrentes (VUs)
        duracion_s:       duración total del test en segundos
        ramp_up_s:        segundos para agregar gradualmente todos los usuarios
    
    Returns:
        Diccionario con todas las métricas del test
    """
    # Listas compartidas entre hilos — protegidas con lock
    tiempos_todos = []   # todos los tiempos de respuesta
    errores_todos = []   # errores registrados
    _lock = threading.Lock()
    
    fin_del_test = time.time() + duracion_s  # cuándo termina el test
    
    def simular_usuario(usuario_id: int):
        """
        Función que ejecuta UN usuario virtual.
        Hace peticiones en loop hasta que el test termine.
        """
        while time.time() < fin_del_test:
            inicio = time.perf_counter()
            
            # TODO: Llama a la función de endpoint y maneja el error si status != 200
            # Pista: usa try/except para capturar excepciones inesperadas
            # y registra el tiempo y si fue error en las listas compartidas
            # ↓ TU CÓDIGO AQUÍ ↓
            pass
            # ↑ TU CÓDIGO AQUÍ ↑
            
            # Pequeña pausa para simular el 'think time' del usuario
            time.sleep(random.uniform(0.1, 0.5))
    
    # Crear y lanzar los hilos (usuarios virtuales)
    # Hacemos ramp-up gradual: no lanzamos todos a la vez
    hilos = []
    inicio_test = time.time()
    
    for i in range(n_usuarios):
        # Retardo de ramp-up: distribuir los usuarios a lo largo de ramp_up_s
        tiempo_espera = (ramp_up_s / n_usuarios) * i
        time.sleep(tiempo_espera / n_usuarios)  # sleep proporcional
        
        hilo = threading.Thread(target=simular_usuario, args=(i,), daemon=True)
        hilos.append(hilo)
        hilo.start()
    
    # Esperar a que todos los hilos terminen
    for hilo in hilos:
        hilo.join(timeout=duracion_s + 10)  # timeout de seguridad
    
    duracion_real = time.time() - inicio_test
    
    # Calcular estadísticas finales
    if not tiempos_todos:
        return {'error': 'No se recolectaron datos — revisa la implementación de simular_usuario()'}
    
    tiempos_todos.sort()
    n_total = len(tiempos_todos)
    n_errores = len(errores_todos)
    
    return {
        'n_usuarios_vus': n_usuarios,
        'duracion_s': round(duracion_real, 1),
        'n_peticiones': n_total,
        'n_errores': n_errores,
        'error_rate': round(n_errores / max(1, n_total) * 100, 2),
        'throughput_rps': round(n_total / duracion_real, 1),
        'p50_ms': round(tiempos_todos[int(n_total * 0.50)], 1),
        'p90_ms': round(tiempos_todos[int(n_total * 0.90)], 1),
        'p95_ms': round(tiempos_todos[int(n_total * 0.95)], 1),
        'p99_ms': round(tiempos_todos[min(int(n_total * 0.99), n_total-1)], 1),
        'max_ms': round(max(tiempos_todos), 1),
        'tiempos_raw': tiempos_todos
    }


print('✅ Función load_test() definida.')
print('   Completa el TODO en simular_usuario() antes de continuar.')

In [ ]:
# ══════════════════════════════════════════════════════════════
#  CELDA 5 — Ejecutar el Load Test
#
#  Aplicamos carga al endpoint de notas durante 60 segundos
#  con 50 usuarios concurrentes.
# ══════════════════════════════════════════════════════════════

print('🏋️ Iniciando Load Test...')
print('   Configuración: 50 VUs | 60 segundos | ramp-up 10s')
print('   Endpoint: GET /api/notas/{id}')
print()

api.reset_stats()

# Ejecutar el load test
# TODO: Llama a load_test() con los parámetros indicados arriba
# Guarda el resultado en una variable llamada 'resultado_load'
# ↓ TU CÓDIGO AQUÍ ↓
resultado_load = None  # reemplaza esto
# ↑ TU CÓDIGO AQUÍ ↑

if resultado_load and 'error' not in resultado_load:
    print('📊 RESULTADOS DEL LOAD TEST')
    print(f'   VUs:          {resultado_load["n_usuarios_vus"]}')
    print(f'   Duración:     {resultado_load["duracion_s"]}s')
    print(f'   Peticiones:   {resultado_load["n_peticiones"]}')
    print(f'   Error rate:   {resultado_load["error_rate"]}%  {"✅" if resultado_load["error_rate"] < 1 else "❌"}')
    print(f'   Throughput:   {resultado_load["throughput_rps"]} req/s')
    print(f'   p50:          {resultado_load["p50_ms"]}ms')
    print(f'   p95:          {resultado_load["p95_ms"]}ms  {"✅ CUMPLE SLA" if resultado_load["p95_ms"] < 500 else "❌ VIOLA SLA"}')
    print(f'   p99:          {resultado_load["p99_ms"]}ms')
    print(f'   Max:          {resultado_load["max_ms"]}ms')
else:
    print('⚠️  Completa el TODO de la celda anterior primero.')

In [ ]:
# ══════════════════════════════════════════════════════════════
#  CELDA 6 — Comparar Baseline vs Load Test (gráfica)
# ══════════════════════════════════════════════════════════════

# TODO: Crea una gráfica de barras comparando los percentiles
# del baseline vs el load test para el endpoint de notas.
#
# La gráfica debe mostrar: p50, p90, p95, p99
# Incluye una línea horizontal roja en 500ms (el SLA)
#
# Pista: usa matplotlib.pyplot
# ↓ TU CÓDIGO AQUÍ ↓
print('TODO: Implementa la gráfica comparativa')
# ↑ TU CÓDIGO AQUÍ ↑

---
# 🔥 PARTE 2 — Stress Testing (20 minutos)

## Objetivo
Encontrar el **punto de quiebre** del sistema — cuántos usuarios concurrentes puede manejar antes de que el rendimiento se degrade inaceptablemente.

## Tu tarea
1. Ejecutar el sistema con cargas crecientes: 10 → 50 → 100 → 200 → 300 VUs
2. Registrar el p95 en cada nivel
3. Identificar en qué punto supera el SLA de 500ms
4. Generar la **curva de capacidad** del sistema

In [ ]:
# ══════════════════════════════════════════════════════════════
#  CELDA 7 — Stress Test: aumentar carga progresivamente
# ══════════════════════════════════════════════════════════════

def stress_test_progresivo(
    funcion_endpoint,
    niveles_vus: list,
    duracion_por_nivel_s: float = 30
) -> list:
    """
    Ejecuta el stress test aumentando la carga en cada nivel.
    
    Args:
        funcion_endpoint:    función del endpoint a estresar
        niveles_vus:         lista de cantidades de usuarios a probar
        duracion_por_nivel_s: cuántos segundos en cada nivel
    
    Returns:
        Lista de resultados por nivel
    """
    resultados_niveles = []
    
    for n_vus in niveles_vus:
        print(f'\n   Probando con {n_vus} VUs...', end=' ')
        
        # TODO: Llama a load_test() con el número de VUs actual
        # Guarda el resultado en 'resultado'
        # ↓ TU CÓDIGO AQUÍ ↓
        resultado = None  # reemplaza esto
        # ↑ TU CÓDIGO AQUÍ ↑
        
        if resultado and 'error' not in resultado:
            resultado['n_vus'] = n_vus
            resultados_niveles.append(resultado)
            estado = '✅' if resultado['p95_ms'] < 500 else '❌'
            print(f'p95={resultado["p95_ms"]}ms {estado} | err={resultado["error_rate"]}% | {resultado["throughput_rps"]} req/s')
        else:
            print('⚠️ Sin datos — verifica el TODO de load_test()')
    
    return resultados_niveles


print('🔥 Iniciando Stress Test progresivo...')
print('   Niveles: 10 → 50 → 100 → 200 → 300 VUs')
print('   Endpoint: GET /api/notas/{id}')
print('   Duración por nivel: 30 segundos')
print()

niveles = [10, 50, 100, 200, 300]

# TODO: Llama a stress_test_progresivo() con los niveles definidos
# Guarda el resultado en 'resultados_stress'
# ↓ TU CÓDIGO AQUÍ ↓
resultados_stress = []
# ↑ TU CÓDIGO AQUÍ ↑

In [ ]:
# ══════════════════════════════════════════════════════════════
#  CELDA 8 — Curva de capacidad (gráfica del stress test)
# ══════════════════════════════════════════════════════════════

# TODO: Crea dos gráficas en un subplot (1 fila, 2 columnas):
#
# Gráfica 1 (izquierda): VUs vs p95 Response Time
#   - Línea de tendencia del p95
#   - Línea roja horizontal en 500ms (SLA)
#   - Área verde = zona donde cumple SLA
#   - Área roja = zona donde viola SLA
#
# Gráfica 2 (derecha): VUs vs Throughput (req/s)
#   - Línea de throughput
#   - Mostrar dónde el throughput deja de crecer (saturación)
#
# ↓ TU CÓDIGO AQUÍ ↓
print('TODO: Implementa la curva de capacidad')
# ↑ TU CÓDIGO AQUÍ ↑

---
# 📋 PARTE 3 — Análisis y Reporte (15 minutos)

## Objetivo
Convertir los números en **conclusiones accionables** para el equipo de desarrollo.

## Tu tarea
1. Identificar el endpoint más lento
2. Diagnosticar la causa (con la información disponible)
3. Escribir el resumen ejecutivo
4. Responder: ¿está listo para producción?

In [ ]:
# ══════════════════════════════════════════════════════════════
#  CELDA 9 — Comparar TODOS los endpoints bajo carga
# ══════════════════════════════════════════════════════════════

print('📊 Midiendo TODOS los endpoints bajo carga (50 VUs)...')
print()

endpoints_bajo_carga = {
    'GET /api/health':          lambda: api.get_health(),
    'GET /api/estudiantes':     lambda: api.listar_estudiantes(),
    'GET /api/notas/{id}':      lambda: api.consultar_notas_estudiante('2024-001'),
    'GET /api/notas/buscar':    lambda: api.buscar_notas('pruebas de software'),
    'POST /api/notas':          lambda: api.registrar_nota('2024-001', 'INF-701', 4.5),
    'GET /api/reportes/{sem}':  lambda: api.generar_reporte('2024-2'),
}

# TODO: Para cada endpoint en endpoints_bajo_carga,
# ejecuta un load_test() con 50 VUs y 30 segundos.
# Guarda los resultados en un diccionario llamado 'resultados_carga_completa'
# con el nombre del endpoint como clave.
#
# Imprime los resultados indicando claramente cuáles cumplen
# y cuáles violan el SLA de 500ms.
#
# ↓ TU CÓDIGO AQUÍ ↓
resultados_carga_completa = {}
print('TODO: Implementa la comparación de todos los endpoints')
# ↑ TU CÓDIGO AQUÍ ↑

In [ ]:
# ══════════════════════════════════════════════════════════════
#  CELDA 10 — Gráfica comparativa de todos los endpoints
# ══════════════════════════════════════════════════════════════

# TODO: Crea una gráfica de barras horizontal comparando el p95
# de todos los endpoints medidos.
#
# - Barras verdes para endpoints que cumplen el SLA
# - Barras rojas para endpoints que violan el SLA
# - Línea vertical roja en 500ms (el SLA)
# - Etiquetas con el valor de p95 en cada barra
#
# ↓ TU CÓDIGO AQUÍ ↓
print('TODO: Implementa la gráfica comparativa de endpoints')
# ↑ TU CÓDIGO AQUÍ ↑

In [ ]:
# ══════════════════════════════════════════════════════════════
#  CELDA 11 — Resumen Ejecutivo
# ══════════════════════════════════════════════════════════════

# TODO: Completa el resumen ejecutivo con los datos reales
# de tus mediciones.
#
# El reporte debe responder:
# 1. ¿Cuántos VUs aguanta antes de violar el SLA?
# 2. ¿Cuáles endpoints violan el SLA de 500ms?
# 3. ¿Cuál es la causa más probable del cuello de botella?
# 4. ¿Está listo para producción? ¿Por qué?
# 5. ¿Qué recomendaciones técnicas harías?

resumen = """
╔══════════════════════════════════════════════════════════════╗
║  RESUMEN EJECUTIVO — PERFORMANCE TEST REPORT                ║
║  Sistema: API Sistema de Notas Universitario                 ║
╠══════════════════════════════════════════════════════════════╣
║  Fecha: {fecha}                                              ║
║  Elaborado por: {nombre}                                     ║
╠══════════════════════════════════════════════════════════════╣
║  MÉTRICAS BAJO CARGA (50 VUs / 60 segundos)                  ║
║                                                              ║
║  Throughput:  TODO req/s                                     ║
║  Error rate:  TODO %                                         ║
║  p95 global:  TODO ms                                        ║
║                                                              ║
╠══════════════════════════════════════════════════════════════╣
║  BREAKING POINT (Stress Test)                                ║
║  Punto de quiebre: TODO VUs                                  ║
║                                                              ║
╠══════════════════════════════════════════════════════════════╣
║  ENDPOINTS PROBLEMÁTICOS                                     ║
║  1. TODO endpoint → p95=TODOms (TODO% sobre SLA)             ║
║  2. TODO endpoint → p95=TODOms (TODO% sobre SLA)             ║
║                                                              ║
╠══════════════════════════════════════════════════════════════╣
║  CAUSA RAÍZ HIPÓTESIS                                        ║
║  TODO: explica qué probablemente causa la lentitud           ║
║                                                              ║
╠══════════════════════════════════════════════════════════════╣
║  VEREDICTO: ¿LISTO PARA PRODUCCIÓN?                          ║
║  TODO: SÍ / NO — y por qué                                   ║
║                                                              ║
╠══════════════════════════════════════════════════════════════╣
║  RECOMENDACIONES                                             ║
║  1. TODO                                                     ║
║  2. TODO                                                     ║
║  3. TODO                                                     ║
╚══════════════════════════════════════════════════════════════╝
"""

from datetime import date
print(resumen.format(fecha=str(date.today()), nombre='Tu nombre aquí'))

---
## 📚 Preguntas de reflexión

Responde las siguientes preguntas en esta celda de texto:

1. **¿Por qué el endpoint `/api/notas/buscar` es más lento que `/api/notas/{id}`?** ¿Qué solución técnica propondrías?

2. **¿Por qué no es suficiente usar solo el promedio para evaluar el rendimiento?** Da un ejemplo con los datos que obtuviste.

3. **Si el equipo pide ejecutar este test en el pipeline de CI/CD, ¿qué cambiarías** en la configuración (duración, VUs, criterio de fallo)?

4. **¿Cuál es la diferencia entre los resultados del load test y el stress test?** ¿Qué información adicional aporta cada uno?